In [3]:
import pandas as pd
from pathlib import Path

out_dir = Path("output")

# 1. Convert Model Catalog to CSV
catalog_path = out_dir / "model_catalog.parquet"
if catalog_path.exists():
    df_cat = pd.read_parquet(catalog_path)
    df_cat.to_csv(out_dir / "model_catalog.csv", index=False)
    print("✅ Exported: output/model_catalog.csv")

# 2. Convert Key Blotters (Run 235 Outlier vs Run 221 Baseline Control)
runs = [
    "model_P14_Breadth_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90",  # Run 235 Outlier
    "model_P12_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5_s42_ep_78",  # Run 221 Gen 13 Champion
    "model_P14_Breadth_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5_s42_ep_82",  # Run 233 Degraded Control
]

for r in runs:
    p = out_dir / f"blotter_df_{r}.parquet"
    if p.exists():
        df = pd.read_parquet(p)
        df.to_csv(out_dir / f"blotter_df_{r}.csv", index=True)
        print(f"✅ Exported: output/blotter_df_{r}.csv")

✅ Exported: output/model_catalog.csv
✅ Exported: output/blotter_df_model_P14_Breadth_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90.csv
✅ Exported: output/blotter_df_model_P12_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5_s42_ep_78.csv
✅ Exported: output/blotter_df_model_P14_Breadth_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5_s42_ep_82.csv


In [1]:
# Quick One-Liner to update macro_df on disk without full data rebuild
from core.paths import LOCAL_DATA_DIR
from core.settings import TradingConfig
from data_pipeline.builder import MacroFeaturePipeline
from data_pipeline.loader import load_processed_data

data = load_processed_data()
cfg = TradingConfig()

print("Recalculating 13-column macro_df with breadth signals...")
new_macro_df = MacroFeaturePipeline.process(
    df_ohlcv=data.df_ohlcv,
    df_indices=getattr(data, "df_indices", None),
    df_fed=getattr(data, "df_fed", None),
    config=cfg,
)

assert new_macro_df.shape[1] == 13, f"Expected 13 columns, got {new_macro_df.shape[1]}"
print(
    f"✅ macro_df shape verified: {new_macro_df.shape}. Columns: {list(new_macro_df.columns[-3:])}"
)

# Save back to processed storage
macro_path = LOCAL_DATA_DIR / "macro_df.parquet"
new_macro_df.to_parquet(macro_path)
print(f"💾 Updated macro_df successfully saved to: {macro_path}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Recalculating 13-column macro_df with breadth signals...
✅ macro_df shape verified: (7466, 13). Columns: ['Breadth_Above_SMA50', 'Breadth_Mom_Spread_21d', 'Breadth_CS_Dispersion_21d']
💾 Updated macro_df successfully saved to: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data\macro_df.parquet


In [ ]:
import numpy as np
import pandas as pd
from core.settings import TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.screener import UniverseScreener

print("=" * 65)
print("[AUDIT] SOURCING PROCESSED FEATURES FOR INVARIANT VERIFICATION")
print("=" * 65)

# 1. Load processed artifacts via centralized loader
try:
    data_bundle = load_processed_data()
    # Support tuple or dict return signatures
    # if isinstance(data_bundle, tuple):
    #     df_ohlcv, features_df, macro_df = (
    #         data_bundle[0],
    #         data_bundle[1],
    #         data_bundle[2],
    #     )
    if isinstance(data_bundle, tuple):
        (
            df_ohlcv,
            macro_df,
            features_df,
        ) = (
            data_bundle[0],
            data_bundle[1],
            data_bundle[2],
        )
    elif isinstance(data_bundle, dict):
        df_ohlcv = data_bundle["df_ohlcv"]
        features_df = data_bundle["features_df"]
        macro_df = data_bundle["macro_df"]
except Exception as err:
    print(
        f"[INFO] loader.load_processed_data() failed: {err}. Attempting direct parquet load..."
    )
    # features_df = pd.read_parquet("output/features_df.parquet")
    # macro_df = pd.read_parquet("output/macro_df.parquet")
    # df_ohlcv = pd.read_parquet("output/df_ohlcv.parquet")
    features_df = pd.read_parquet("data/features_df.parquet")
    macro_df = pd.read_parquet("data/macro_df.parquet")
    df_ohlcv = pd.read_parquet("data/df_ohlcv.parquet")

cfg = TradingConfig()

# -----------------------------------------------------------------------------
# CHECK 1: Schema Invariant & Column Contracts
# -----------------------------------------------------------------------------
required_new_cols = ["Mom_252_21", "SemiDev_63", "Trend_R2_63"]
pruned_cols = ["AutoCorr_15", "Range_Pos_20"]

missing_cols = [c for c in required_new_cols if c not in features_df.columns]
stale_cols = [c for c in pruned_cols if c in features_df.columns]

print(f"\n1. COLUMN PARITY CHECK:")
if missing_cols:
    print(
        f"  ❌ FAILED: Missing new structural columns: {missing_cols}\n"
        f"     ACTION REQUIRED: Re-run 00_RLVR_data_process_v8.ipynb to regenerate features_df!"
    )
    raise AssertionError(f"Stale features_df missing {missing_cols}")
else:
    print(f"  ✅ PASSED: All 3 new columns present -> {required_new_cols}")

if stale_cols:
    print(
        f"  ⚠️  WARNING: Pruned legacy columns still present in features_df: {stale_cols} (Harmless if unused, but re-run 00 recommended)"
    )
else:
    print(f"  ✅ PASSED: Pruned columns successfully purged -> {pruned_cols}")

# -----------------------------------------------------------------------------
# CHECK 2: Numerical & Boundary Sanity on New Factors
# -----------------------------------------------------------------------------
print(f"\n2. NUMERICAL & BOUNDARY CHECKS:")

# A. Trend_R2_63 (Must be bounded strictly in [-1.0, 1.0])
trend_r2 = features_df["Trend_R2_63"].dropna()
min_r2, max_r2 = trend_r2.min(), trend_r2.max()
print(f"  • Trend_R2_63 Bounds: [{min_r2:.4f}, {max_r2:.4f}]")
assert (
    min_r2 >= -1.0001 and max_r2 <= 1.0001
), f"Trend_R2_63 out of bounds [-1, 1]: {min_r2}, {max_r2}"
assert (trend_r2 != 0.0).any(), "Trend_R2_63 is entirely zero!"

# B. SemiDev_63 (Downside deviation must be strictly non-negative >= 0.0)
semidev = features_df["SemiDev_63"].dropna()
min_semi, max_semi = semidev.min(), semidev.max()
median_semi = semidev.median()
print(
    f"  • SemiDev_63 Bounds:  [{min_semi:.6f}, {max_semi:.6f}] | Median: {median_semi:.4f}"
)
assert min_semi >= 0.0, f"SemiDev_63 has negative values: {min_semi}"
assert (semidev != 0.0).any(), "SemiDev_63 is entirely zero!"

# C. Mom_252_21 (Finite, reasonable annual momentum range)
mom_252 = features_df["Mom_252_21"].dropna()
print(
    f"  • Mom_252_21 Range:   [{mom_252.min():.2%}, {mom_252.max():.2%}] | Median: {mom_252.median():.2%}"
)
assert np.isfinite(mom_252).all(), "Mom_252_21 contains Inf/-Inf values!"

print(f"  ✅ PASSED: Statistical distributions verified and healthy.")

# -----------------------------------------------------------------------------
# CHECK 3: End-to-End Observation Assembler & Screener Tripwire
# -----------------------------------------------------------------------------
print(f"\n3. INTEGRATION VERIFICATION: SCREENER -> OBSERVATION PASS-THROUGH")

if isinstance(df_ohlcv.index, pd.MultiIndex):
    df_close = df_ohlcv["Adj Close"].unstack(level=0)
else:
    df_close = df_ohlcv

# Settle calendar on benchmark
bm = cfg.benchmark_ticker
cal = pd.DatetimeIndex(df_close[bm].dropna().index.unique().sort_values())

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=cal,
    config=cfg,
)

# Test observation assembly on the last valid trading date
test_date = cal[-20]
candidates = screener.filter_universe(test_date, cfg.thresholds)
start_date = cal[cal.get_loc(test_date) - cfg.win_63d]

obs = screener.build_observation(test_date, candidates, start_date)

# Verify fields exist on MarketObservation
assert hasattr(obs, "mom_252_21"), "MarketObservation missing 'mom_252_21' attribute!"
assert hasattr(obs, "semidev_63"), "MarketObservation missing 'semidev_63' attribute!"
assert hasattr(obs, "trend_r2_63"), "MarketObservation missing 'trend_r2_63' attribute!"

print(f"  • Test Date:               {test_date.date()}")
print(f"  • Screened Candidates:     {len(candidates)} tickers")
print(f"  • Sample Mom_252_21:       {obs.mom_252_21.iloc[:3].round(4).to_dict()}")
print(f"  • Sample SemiDev_63:       {obs.semidev_63.iloc[:3].round(4).to_dict()}")
print(f"  • Sample Trend_R2_63:      {obs.trend_r2_63.iloc[:3].round(4).to_dict()}")

print("\n" + "=" * 65)
print("🎯 100% INVARIANT VERIFICATION COMPLETE: ALL SYSTEMS READY FOR 01")
print("=" * 65)

In [ ]:
features_df.info()

In [ ]:
import pandas as pd
from core.paths import LOCAL_DATA_DIR
from core.settings import CacheConfig

df_cache = pd.read_parquet(LOCAL_DATA_DIR / CacheConfig.get_filename())
cols = df_cache.columns.tolist()

assert "41d_Momentum (63d)" in cols, "Missing Mom_63!"
assert "41d_Momentum (126d)" in cols, "Missing Mom_126!"
assert "41d_Residual Low-Vol (63d)" in cols, "Missing IVol_63!"
assert not any("Slope" in c for c in cols), "Stale Slope features detected!"
print(f"AlphaCache verified successfully! Shape: {df_cache.shape}")

In [ ]:
import pandas as pd
from core.paths import GLOBAL_PROCESSED_DIR, LOCAL_DATA_DIR
from core.settings import TradingConfig
from data_pipeline.builder import generate_features

print("[1/3] Loading raw data...")
df_ohlcv = pd.read_parquet(GLOBAL_PROCESSED_DIR / "df_ohlcv.parquet")
config = TradingConfig()

print("[2/3] Generating updated features (Mom_63, Mom_126, IVol_63)...")
features_df, macro_df = generate_features(df_ohlcv, config=config)

print("[3/3] Saving to LOCAL_DATA_DIR...")
features_df.to_parquet(LOCAL_DATA_DIR / "features_df.parquet")
macro_df.to_parquet(LOCAL_DATA_DIR / "macro_df.parquet")
print("DONE! features_df.parquet successfully refreshed.")

In [ ]:
"""scripts/heal_blotters.py: Re-evaluates OOS checkpoints with pure geometric accounting."""

import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.validator import AgentEvaluator

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
config = TradingConfig()
benchmark_ticker = config.benchmark_ticker

# Load pipeline
df_ohlcv, macro_df, _ = load_processed_data()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)
feature_cube = pd.read_parquet(LOCAL_DATA_DIR / CacheConfig.get_filename())

reward_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
reward_matrix["CASH"] = 0.0

valid_dates = feature_cube.index.get_level_values("Date").unique()
cal_test = trading_calendar[
    (trading_calendar.isin(valid_dates))
    & (trading_calendar >= pd.Timestamp("2022-04-01"))
]

all_checkpoints = sorted(list(checkpoint_dir.glob("*.pt")))
print(
    f"🔧 Healing {len(all_checkpoints)} Checkpoints with Strict Geometric Accounting..."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for ckpt_path in all_checkpoints:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m_cfg = TradingConfig()
    gp = ckpt.get("grid_params", {})
    if "min_active_tilt" in gp:
        m_cfg.min_active_tilt = float(gp["min_active_tilt"])
    if "min_basket_width" in gp:
        m_cfg.min_basket_width = int(gp["min_basket_width"])

    env_test = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=reward_matrix,
        calendar=cal_test,
        macro_df=macro_df,
        config=m_cfg,
        randomize_start=False,
        episode_steps=0,
    )
    gym_test = RLVRGymEnv(env_test, macro_df)
    gym_test.is_training = False

    s_state = ckpt.get("scaler_state")
    if s_state:
        gym_test.scaler.load_state(s_state)

    agent = AbsoluteZeroAgent(
        obs_dim=gym_test.observation_space.shape[0],
        action_dim=gym_test.action_space.shape[0],
    ).to(device)
    agent.load_state_dict(ckpt.get("model_state_dict", ckpt))

    res = AgentEvaluator.evaluate(
        agent=agent, env=gym_test, device=device, detailed_log=True
    )

    stem = ckpt_path.stem
    blotter_df = pd.DataFrame(res.get("blotter", []))
    blotter_path = OUTPUT_DIR / f"blotter_df_{stem}.parquet"
    pkl_path = OUTPUT_DIR / f"results_{stem}.pkl"

    blotter_df.to_parquet(blotter_path, index=False)
    with open(pkl_path, "wb") as f:
        pickle.dump(res, f)

    # Verification probe on terminal row
    last_agent_eq = blotter_df["agent_equity"].iloc[-1]
    last_bm_eq = blotter_df["bm_equity"].iloc[-1]
    last_alpha_eq = blotter_df["alpha_equity"].iloc[-1]
    expected_alpha = last_agent_eq / last_bm_eq

    assert np.isclose(
        last_alpha_eq, expected_alpha, atol=1e-6
    ), "Geometric invariant check failed!"
    print(
        f"  ✅ {stem}: Agent={last_agent_eq:.4f}, BM={last_bm_eq:.4f}, Geometric Alpha={last_alpha_eq:.6f}"
    )

print("\n🎉 ALL CHECKPOINTS HEALED WITH PURE GEOMETRIC PROVENANCE!")

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Delete truncated blotter parquets
for f in Path("output").glob("blotter_df_*.parquet"):
    f.unlink()
print("🗑️ Stale blotter parquets deleted.")

# 2. Reset Grid_sweep.csv so Cell 4 re-evaluates all checkpoints
csv_path = Path("Grid_sweep.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    # Remove rows from the current sweep to trigger re-evaluation
    df = df[df["Param Sweep"] != "param_grid_sweep"]
    df.to_csv(csv_path, index=False)
    print("🧹 Grid_sweep.csv cleared of stale sweep records.")

In [ ]:
import pandas as pd, torch
from pathlib import Path

# 1. Inspect model_catalog.parquet
cat_path = Path("output/model_catalog.parquet")
if cat_path.exists():
    cat = pd.read_parquet(cat_path)
    print("=== MODEL CATALOG COLUMNS ===")
    print(cat.columns.tolist())
    print("\n=== TOP 2 CATALOG RECORDS ===")
    print(cat.head(2).to_dict(orient="records"))

# 2. Inspect a Pivot 1 checkpoint payload directly
ckpt_path = next(Path("output/model_checkpoints").glob("*.pt"))
payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"\n=== PAYLOAD KEYS for {ckpt_path.name} ===")
print(payload.keys())
print("\n=== GRID_PARAMS CONTENT ===")
print(payload.get("grid_params"))